In [1]:
import os
import numpy as np
import pandas as pd

# Relative to Our Notebooks/
PROVIDED_DIR = "../Provided Datasets"
NEW_DIR = "../New Datasets"
OUTPUT_DIR = NEW_DIR + "/Combined Training"
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Expected training files (we will look in BOTH folders for each file)
EXPECTED_FILES = {
    "gaia": "gaia_features_training.csv",
    "jrc_gsw": "jrc_gsw_features_training.csv",
    "landsat_allbands": "landsat_features_training_allbands.csv",
    "terraclimate_allvars": "terraclimate_features_training_allvars.csv",
    "esa_cci": "esa_cci_features_training.csv",
}

def resolve_path(fname: str) -> str | None:
    """Return the first existing path for fname across Provided and New dirs."""
    for base in (PROVIDED_DIR, NEW_DIR):
        p = os.path.join(base, fname)
        if os.path.exists(p):
            return p
    return None

resolved = {}
missing = []
for key, fname in EXPECTED_FILES.items():
    p = resolve_path(fname)
    if p is None:
        missing.append((key, fname))
    else:
        resolved[key] = p

print("Resolved input paths:")
for k, p in resolved.items():
    print(f" - {k:18s} -> {os.path.abspath(p)}")

if missing:
    msg = "Missing these expected files in BOTH folders:\n" + "\n".join([f"{k}: {f}" for k, f in missing])
    raise FileNotFoundError(msg)

print("\nAll expected files found")

In [3]:
def standardize_join_keys(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize join columns to: latitude, longitude, sample_date."""
    out = df.copy()
    rename_map = {}
    for c in out.columns:
        lc = c.strip().lower()
        if lc in {"latitude", "lat"} or lc.startswith("lat"):
            rename_map[c] = "latitude"
        elif lc in {"longitude", "lon", "lng"} or lc.startswith("lon"):
            rename_map[c] = "longitude"
        # Be conservative: only map obvious sample-date columns
        elif lc in {"sample date", "sample_date"}:
            rename_map[c] = "sample_date"
    out = out.rename(columns=rename_map)

    # If a dataset used a generic "Date" column, map it ONLY if sample_date doesn't exist yet
    if "sample_date" not in out.columns:
        for c in list(out.columns):
            if c.strip().lower() == "date":
                out = out.rename(columns={c: "sample_date"})
                break

    # Drop duplicate columns created by renaming collisions
    out = out.loc[:, ~out.columns.duplicated(keep="first")]

    required = {"latitude", "longitude", "sample_date"}
    missing = required - set(out.columns)
    if missing:
        raise ValueError(f"Missing required join columns after standardization: {missing}")

    # helper parsed date (not used for join)
    out["sample_date_parsed"] = pd.to_datetime(out["sample_date"], errors="coerce", infer_datetime_format=True)
    return out

In [ ]:
# Load + standardize
datasets = {}
for name, path in resolved.items():
    df = pd.read_csv(path)
    df_std = standardize_join_keys(df)
    datasets[name] = df_std
    print(f"{name:18s} shape={df_std.shape}  cols={len(df_std.columns)}")

In [5]:
# Outer merge on join keys
merge_keys = ["latitude", "longitude", "sample_date"]

merged = None
for name, df in datasets.items():
    if merged is None:
        merged = df
    else:
        merged = pd.merge(
            merged,
            df,
            on=merge_keys,
            how="outer",
            suffixes=("", f"__{name}")  # helps prevent _x/_y
        )

def remove_extra_date_columns(df):
    cols_to_drop = [
        c for c in df.columns
        if ("date" in c.lower()) and (c.lower() != "sample_date")
    ]
    
    print("Dropping from dataframe:")
    print(cols_to_drop)
    
    return df.drop(columns=cols_to_drop)

# Apply to all versions you are saving
merged = remove_extra_date_columns(merged)

if "filled_mean" in globals():
    filled_mean = remove_extra_date_columns(filled_mean)

if "dropped_na" in globals():
    dropped_na = remove_extra_date_columns(dropped_na)

print("Final shape:", merged.shape)

Dropping from dataframe:
['sample_date_parsed', 'sample_date_parsed__jrc_gsw', 'sample_date_parsed__landsat_allbands', 'sample_date_parsed__terraclimate_allvars', 'sample_date_parsed__esa_cci']
Final shape: (9319, 48)


In [6]:
# shape checking
esa = pd.read_csv(os.path.join(PROJECT_ROOT + "/New Datasets", "esa_cci_features_training.csv"))
jrc = pd.read_csv(os.path.join(PROJECT_ROOT + "/New Datasets", "jrc_gsw_features_training.csv"))
gaia = pd.read_csv(os.path.join(PROJECT_ROOT + "/New Datasets", "gaia_features_training.csv"))
landsat = pd.read_csv(os.path.join(PROJECT_ROOT + "/New Datasets", "landsat_features_training_allbands.csv"))
terraclimate = pd.read_csv(os.path.join(PROJECT_ROOT + "/New Datasets", "terraclimate_features_training_allvars.csv"))
real_shape = esa.shape[1] + jrc.shape[1] + gaia.shape[1] + landsat.shape[1] + terraclimate.shape[1]
# subtract 15 for the join keys, then add 3 since we need lat long and date
real_shape = real_shape - 15 + 3
print(f"Real Shape: {real_shape}")
print(f"Merged Shape: {merged.shape[1]}")

# combined validation
# Validate that merged cell values match each source dataset
merge_keys = ["latitude", "longitude", "sample_date"]

def _col_in_merged(col: str, dataset_name: str) -> str | None:
    if col in merge_keys:
        return col
    if col in merged.columns:
        return col
    alt = f"{col}__{dataset_name}"
    if alt in merged.columns:
        return alt
    return None


def _equal_series(a: pd.Series, b: pd.Series, tol: float = 1e-6) -> pd.Series:
    # Treat NaN == NaN as equal; compare numerics with tolerance
    a_num = pd.to_numeric(a, errors="coerce")
    b_num = pd.to_numeric(b, errors="coerce")

    both_nan = a.isna() & b.isna()
    both_num = a_num.notna() & b_num.notna()
    close_num = pd.Series(False, index=a.index)
    close_num[both_num] = np.isclose(a_num[both_num], b_num[both_num], atol=tol, rtol=0)

    # Non-numeric or mixed: compare as strings (but keep NaNs handled already)
    a_str = a.astype(str)
    b_str = b.astype(str)
    str_equal = (a_str == b_str) & (~both_num)

    return both_nan | close_num | str_equal


def validate_dataset(name: str, df_std: pd.DataFrame, tol: float = 1e-6) -> pd.DataFrame:
    # Drop helper parsed date if present
    df_std = df_std.drop(columns=[c for c in df_std.columns if c == "sample_date_parsed"], errors="ignore")

    # Align with merged by keys
    df_std = df_std.copy()
    df_std["_row_id"] = df_std.index
    merged_with = merged.merge(df_std, on=merge_keys, how="left", suffixes=("", "__src"))

    mismatch_rows = []
    for col in df_std.columns:
        if col in merge_keys or col == "_row_id":
            continue
        merged_col = _col_in_merged(col, name)
        if merged_col is None:
            mismatch_rows.append({"dataset": name, "column": col, "issue": "missing_in_merged"})
            continue

        src_col = f"{col}__src"
        if src_col not in merged_with.columns:
            mismatch_rows.append({"dataset": name, "column": col, "issue": "missing_in_source_after_merge"})
            continue

        equal_mask = _equal_series(merged_with[merged_col], merged_with[src_col], tol=tol)
        if not bool(equal_mask.all()):
            mismatch_rows.append({
                "dataset": name,
                "column": col,
                "issue": "value_mismatch",
                "mismatch_count": int((~equal_mask).sum()),
            })

    return pd.DataFrame(mismatch_rows)


mismatch_reports = []
for name, df_std in datasets.items():
    report = validate_dataset(name, df_std)
    mismatch_reports.append(report)

mismatch_summary = pd.concat(mismatch_reports, ignore_index=True) if mismatch_reports else pd.DataFrame()
print("Mismatch summary (empty means all checks passed):")
display(mismatch_summary)

Real Shape: 48
Merged Shape: 48
Mismatch summary (empty means all checks passed):


""


In [13]:
# dropping features
dropped_cols = [
    "qa_pixel",
    "qa_aerosol",
    "esa_processed_flag",
    "esa_observation_count",
    "esa_current_pixel_state",
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus",
    "coastal",
    "lwir11"
]

cols_to_drop = [c for c in dropped_cols if c in merged.columns]
filtered_no_pixel_flags = merged.drop(columns=cols_to_drop).copy()

out_no_flags = os.path.join(OUTPUT_DIR, "combined_training_dataset.csv")
filtered_no_pixel_flags.to_csv(out_no_flags, index=False)

print("Dropped columns:", cols_to_drop)
print("Saved:", os.path.abspath(out_no_flags))
# should be 38, we cut 5 flags, 2 bands, andthe 3 target variables
print(filtered_no_pixel_flags.shape)

Dropped columns: ['qa_pixel', 'qa_aerosol', 'esa_processed_flag', 'esa_observation_count', 'esa_current_pixel_state', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'coastal', 'lwir11']
Saved: /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/Combined Training/combined_training_dataset.csv
(9319, 38)


In [11]:
# examining the nulls
# Number of nulls in each column
null_counts = filtered_no_pixel_flags.isna().sum()

# Convert to DataFrame for nicer display
null_summary = (
    null_counts
    .reset_index()
    .rename(columns={"index": "column", 0: "null_count"})
    .sort_values("null_count", ascending=False)
)

# Add percentage null
null_summary["percent_null"] = (
    100 * null_summary["null_count"] / len(merged)
).round(2)

# Show full table
pd.set_option("display.max_rows", None)
null_summary

# save a dataset of rows where any Landsat band/index is null (only keep those columns)
landsat_targets = ["nir","ndmi","blue","red","swir22","swir16","green","mndwi",]
merged_cols_lower = {c.lower(): c for c in filtered_no_pixel_flags.columns}

resolved_cols = []
for t in landsat_targets:
    if t in merged_cols_lower:
        resolved_cols.append(merged_cols_lower[t])
        continue
    alt = f"{t}__landsat"
    if alt in merged_cols_lower:
        resolved_cols.append(merged_cols_lower[alt])
        continue

missing = [t for t, c in zip(landsat_targets, resolved_cols + [None] * (len(landsat_targets) - len(resolved_cols))) if c is None]
if missing:
    raise ValueError(f"Missing expected Landsat columns in merged: {missing}")

null_mask = filtered_no_pixel_flags[resolved_cols].isna().any(axis=1)
landsat_nulls = filtered_no_pixel_flags.loc[null_mask, resolved_cols].copy()
out_path = os.path.join(PROJECT_ROOT, "landsat_nulls.csv")
landsat_nulls.to_csv(out_path, index=False)

In [12]:
# Correlation matrix + highly correlated feature pairs

numeric_cols = filtered_no_pixel_flags.select_dtypes(include=[np.number]).columns
corr = filtered_no_pixel_flags[numeric_cols].corr()

# Full correlation matrix (can be large)
print("Correlation matrix shape:", corr.shape)
display(corr)

# List top absolute correlations (excluding self-correlation)
abs_corr = corr.abs()
np.fill_diagonal(abs_corr.values, 0)

pairs = (
    abs_corr.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b", 0: "abs_corr"})
)

# Drop duplicate pairs (A,B) vs (B,A)
pairs = pairs[pairs["feature_a"] < pairs["feature_b"]]

# Show pairs above threshold
threshold = 0.9
high_corr = pairs[pairs["abs_corr"] >= threshold].sort_values("abs_corr", ascending=False)
print(f"Highly correlated pairs (|r| >= {threshold}): {len(high_corr)}")
display(high_corr)


Correlation matrix shape: (37, 37)


,latitude,longitude,gaia_changed_ever_frac,gaia_impervious_frac_by_sample_year,gaia_recent_change_5y_frac,gaia_years_since_change_mean,gaia_transition_year_mean_changed_pixels,gsw_change,gsw_extent,gsw_occurrence,...,swe,srad,tmax,tmin,vap,vpd,ws,pdsi,esa_lccs_class,esa_change_count
latitude,1.000000,0.624468,-0.008186,-0.008266,0.034533,-0.008151,-0.008176,0.054291,-0.011862,0.029326,...,NaN,0.135378,0.295999,0.005685,0.026982,0.304922,-0.412407,-0.016319,0.287132,-0.074657
longitude,0.624468,1.000000,-0.041832,-0.038242,-0.010196,-0.043382,-0.041804,-0.002020,0.017278,-0.055236,...,NaN,-0.070313,0.096004,0.052798,0.221191,-0.134730,-0.385840,-0.026427,0.022890,0.023355
gaia_changed_ever_frac,-0.008186,-0.041832,1.000000,0.998205,0.661813,0.990930,0.999999,0.114104,-0.109352,-0.076120,...,NaN,0.038434,-0.008537,-0.047261,-0.066127,0.039060,0.023908,-0.023213,0.067305,-0.009109
gaia_impervious_frac_by_sample_year,-0.008266,-0.038242,0.998205,1.000000,0.662181,0.993541,0.998177,0.112587,-0.107954,-0.075351,...,NaN,0.038596,-0.009516,-0.048570,-0.067417,0.038813,0.024284,-0.029686,0.068731,-0.007191
gaia_recent_change_5y_frac,0.034533,-0.010196,0.661813,0.662181,1.000000,0.636831,0.662120,0.071262,-0.071484,-0.056023,...,NaN,0.037761,0.018274,-0.025316,-0.042876,0.053419,0.034876,-0.063191,0.065382,-0.038268
gaia_years_since_change_mean,-0.008151,-0.043382,0.990930,0.993541,0.636831,1.000000,0.990800,0.108686,-0.102661,-0.069395,...,NaN,0.038955,-0.004474,-0.044594,-0.064634,0.043620,0.023044,-0.031148,0.070470,-0.000922
gaia_transition_year_mean_changed_pixels,-0.008176,-0.041804,0.999999,0.998177,0.662120,0.990800,1.000000,0.114146,-0.109407,-0.076178,...,NaN,0.038436,-0.008564,-0.047282,-0.066132,0.039020,0.023926,-0.023269,0.067264,-0.009191
gsw_change,0.054291,-0.002020,0.114104,0.112587,0.071262,0.108686,0.114146,1.000000,-0.953432,-0.708693,...,NaN,-0.003207,-0.040184,-0.065226,-0.040593,-0.044926,-0.050334,-0.012364,-0.121997,0.041532
gsw_extent,-0.011862,0.017278,-0.109352,-0.107954,-0.071484,-0.102661,-0.109407,-0.953432,1.000000,0.795522,...,NaN,0.008465,0.041264,0.049693,0.019750,0.059834,0.036064,0.020265,0.208174,-0.041708
gsw_occurrence,0.029326,-0.055236,-0.076120,-0.075351,-0.056023,-0.069395,-0.076178,-0.708693,0.795522,1.000000,...,NaN,0.052554,0.050413,0.001986,-0.077416,0.156194,0.050841,0.023486,0.378570,-0.067812


Highly correlated pairs (|r| >= 0.9): 16


,feature_a,feature_b,abs_corr
78,gaia_changed_ever_frac,gaia_transition_year_mean_changed_pixels,0.999999
75,gaia_changed_ever_frac,gaia_impervious_frac_by_sample_year,0.998205
114,gaia_impervious_frac_by_sample_year,gaia_transition_year_mean_changed_pixels,0.998177
113,gaia_impervious_frac_by_sample_year,gaia_years_since_change_mean,0.993541
77,gaia_changed_ever_frac,gaia_years_since_change_mean,0.990930
221,gaia_transition_year_mean_changed_pixels,gaia_years_since_change_mean,0.990800
662,blue,green,0.980015
521,green,red,0.974154
665,blue,red,0.972030
335,gsw_occurrence,gsw_seasonality,0.960894


In [10]:
# Save merged + two filled versions
raw_out = os.path.join(OUTPUT_DIR, "combined_training_dataset.csv")
merged.to_csv(raw_out, index=False)

# Version 1: Drop rows with ANY null values
dropped_na = merged.dropna()

drop_out = os.path.join(OUTPUT_DIR, "combined_training_dataset_dropna.csv")
dropped_na.to_csv(drop_out, index=False)

print("Wrote:", os.path.abspath(drop_out))
print("Original rows:", len(merged))
print("Rows after dropna:", len(dropped_na))


# Version 2: numeric nulls -> column mean
filled_mean = merged.copy()
numeric_cols = filled_mean.select_dtypes(include=[np.number]).columns
for c in numeric_cols:
    m = filled_mean[c].mean()
    if pd.notna(m):
        filled_mean[c] = filled_mean[c].fillna(m)
mean_out = os.path.join(OUTPUT_DIR, "combined_training_dataset_nulls_as_mean.csv")
filled_mean.to_csv(mean_out, index=False)

print("Wrote:")
print(" -", os.path.abspath(raw_out))
print(" -", os.path.abspath(drop_out))
print(" -", os.path.abspath(mean_out))

Wrote: /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/Combined Training/combined_training_dataset_dropna.csv
Original rows: 9319
Rows after dropna: 2901
Wrote:
 - /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/Combined Training/combined_training_dataset.csv
 - /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/Combined Training/combined_training_dataset_dropna.csv
 - /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/Combined Training/combined_training_dataset_nulls_as_mean.csv
